# 02 — Feature engineering walkthrough

Build the engineered feature frame from raw transactions, inspect the per-feature distributions by class, and verify the zero-leakage guarantee. The output of this notebook is what `src.models.train` consumes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.loader import LABEL_COLUMN, DataLoader
from src.features.pipelines import build_engineered_frame

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

In [ ]:
loader = DataLoader()
frame = loader.load_sample(n=200_000, random_state=42)
print(f'Sample size: {len(frame):,}')
print(f'Positive rate preserved by stratified sampler: {frame[LABEL_COLUMN].mean():.6f}')

## 1. Build the engineered frame

The orchestrator calls each feature builder in the correct order. On 200k rows this typically completes in 20–40 seconds; the full 5M-row dataset takes ~15 minutes.

In [ ]:
bundle = build_engineered_frame(frame)
print(f'Engineered columns: {len(bundle.numerical_columns) + len(bundle.categorical_columns)}')
print(f'  Numerical: {len(bundle.numerical_columns)}')
print(f'  Categorical: {len(bundle.categorical_columns)}')
print('\nSample numerical features:')
print(bundle.numerical_columns[:10])

## 2. Distributional comparison: by class

For features that should carry signal, the illicit-class distribution should visibly differ from the licit-class distribution.

In [ ]:
signal_features = [
    'src_24h_sub_threshold_share',
    'src_24h_round_amount_share',
    'entity_in_out_amount_ratio_24h',
    'edge_novelty_24h',
]
available = [f for f in signal_features if f in bundle.frame.columns]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, feature in zip(axes.ravel(), available):
    for label, colour in [(0, '#0f172a'), (1, '#ef4444')]:
        values = bundle.frame.loc[bundle.frame[LABEL_COLUMN] == label, feature].dropna()
        if len(values) > 0:
            ax.hist(values, bins=40, alpha=0.5, color=colour, label=f'class={label}', density=True)
    ax.set_title(feature)
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Zero-leakage verification

All rolling features are computed with `closed='left'`. The first transaction for any entity should have NaN aggregates because there is no prior history. Verify with a spot check.

In [ ]:
spot_features = [c for c in bundle.numerical_columns if c.startswith('src_24h_')]
first_per_entity = bundle.frame.groupby('Account').head(1)[spot_features]
print('NaN share on first transaction per entity (count features):')
for col in spot_features[:6]:
    print(f'  {col}: {first_per_entity[col].isna().mean():.4f}')
print('\n(Count features are filled with 0 by the engineering layer; mean/std/max remain NaN as expected.)')

## 4. Feature correlation

High intra-family correlation is expected (1h is correlated with 24h windows of the same statistic). The model handles correlated features via regularisation; we surface the correlation matrix as a visual sanity check.

In [ ]:
subset = [c for c in bundle.numerical_columns if 'src_' in c and 'amount' in c][:8]
if len(subset) >= 2:
    corr = bundle.frame[subset].corr()
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, cbar_kws={'shrink': 0.8})
    plt.title('Correlation among amount features (source side)')
    plt.tight_layout()
    plt.show()